# Tennis Events Demo

Use the dropdown to pick a sample clip from `data/samples/`, then click **Render events**.

The notebook will:
- run or reuse the tracked source video and canonical `match_events.json`
- build a tennis-aware bird's-eye video from the same tracking
- overlay detected events on both videos
- show a compact event list for quick inspection

All event artifacts are cached per video in `out/events/`.


In [1]:
from __future__ import annotations

import html
import json
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import Video, display

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tennis_tracker.birdseye import render_birdseye_video
from tennis_tracker.events import (
    ensure_match_events_for_video,
    render_event_annotation_video,
    slugify_video_name,
    summarize_match_events,
)

SAMPLES_DIR = ROOT / "data" / "samples"
SAMPLE_VIDEOS = sorted(SAMPLES_DIR.glob("*.mp4"))
assert SAMPLE_VIDEOS, f"No sample videos found in {SAMPLES_DIR}"

DEFAULT_VIDEO = next((p for p in SAMPLE_VIDEOS if p.name == "broadcast_2s.mp4"), SAMPLE_VIDEOS[0])
WEIGHTS_PATH = ROOT / "models" / "tracknet_weights.pth"
EVENTS_OUT_DIR = ROOT / "out" / "events"
EVENTS_OUT_DIR.mkdir(parents=True, exist_ok=True)

assert WEIGHTS_PATH.exists(), f"Missing TrackNet weights: {WEIGHTS_PATH}"

selector = widgets.Dropdown(
    options=[(video.name, str(video)) for video in SAMPLE_VIDEOS],
    value=str(DEFAULT_VIDEO),
    description="Video:",
    layout=widgets.Layout(width="430px"),
)
force_rerun = widgets.Checkbox(value=False, description="Force rerun")
render_button = widgets.Button(description="Render events", button_style="primary")
ui_output = widgets.Output()


def build_video_panel(video_path: Path, title: str) -> widgets.HTML:
    video = Video(
        filename=str(video_path),
        embed=True,
        width=560,
        html_attributes="controls autoplay loop muted playsinline",
    )
    video_html = video._repr_html_() or ""
    return widgets.HTML(
        value=(
            "<div style='width:100%'>"
            f"<div style='font-weight:600; margin-bottom:6px;'>{title}</div>"
            f"{video_html}"
            "</div>"
        ),
        layout=widgets.Layout(width="50%"),
    )


def is_stale(output_path: Path, *input_paths: Path) -> bool:
    if not output_path.exists():
        return True
    output_mtime = output_path.stat().st_mtime
    return any(path.exists() and path.stat().st_mtime > output_mtime for path in input_paths)


def event_lines(events_json_path: Path) -> str:
    data = json.loads(events_json_path.read_text())
    lines: list[str] = []
    for rally in data.get("rallies", []):
        summary = rally.get("summary", {})
        lines.append(
            f"Rally {rally['id']} | frames {rally['start_frame']}-{rally['end_frame']} | "
            f"serve={summary.get('serve_actor', 'unknown')} | hits={summary.get('hit_count', 0)} | "
            f"bounces={summary.get('bounce_count', 0)}"
        )
        for event in rally.get("events", []):
            lines.append(
                f"  f={event['frame']:>3} | {event['type']:<11} | {event.get('actor', 'unknown'):<11} | "
                f"conf={float(event.get('confidence', 0.0)):.2f}"
            )
    return "\n".join(lines) if lines else "No rally events detected."


def render_event_demo(video_path: Path, *, force: bool = False) -> None:
    slug = slugify_video_name(video_path.name)
    artifacts = ensure_match_events_for_video(
        video_path=video_path,
        output_dir=EVENTS_OUT_DIR,
        weights_path=WEIGHTS_PATH,
        force=force,
    )

    birdseye_video_path = EVENTS_OUT_DIR / f"{slug}_birdseye.mp4"
    source_events_video_path = EVENTS_OUT_DIR / f"{slug}_events_source.mp4"
    birdseye_events_video_path = EVENTS_OUT_DIR / f"{slug}_events_birdseye.mp4"

    if force or is_stale(birdseye_video_path, artifacts["tracking_json"], artifacts["events_json"]):
        render_birdseye_video(
            tracking_json_path=artifacts["tracking_json"],
            output_video_path=birdseye_video_path,
            events_json_path=artifacts["events_json"],
            player_style="dot",
            ball_strategy="event_aware",
        )

    if force or is_stale(source_events_video_path, artifacts["tracking_video"], artifacts["events_json"]):
        render_event_annotation_video(
            input_video_path=artifacts["tracking_video"],
            events_json_path=artifacts["events_json"],
            output_video_path=source_events_video_path,
        )

    if force or is_stale(birdseye_events_video_path, birdseye_video_path, artifacts["events_json"]):
        render_event_annotation_video(
            input_video_path=birdseye_video_path,
            events_json_path=artifacts["events_json"],
            output_video_path=birdseye_events_video_path,
        )

    summary = summarize_match_events(artifacts["events_json"])
    summary["events_json"] = str(artifacts["events_json"].relative_to(ROOT))

    ui_output.clear_output(wait=True)
    with ui_output:
        print(summary)
        display(
            widgets.HBox(
                [
                    build_video_panel(source_events_video_path, "Tracked source + events"),
                    build_video_panel(birdseye_events_video_path, "Bird's-eye + events (tennis-aware ball path)"),
                ],
                layout=widgets.Layout(align_items="flex-start"),
            )
        )
        display(
            widgets.HTML(
                value=(
                    "<div style='margin-top:12px; font-weight:600;'>Detected events</div>"
                    f"<pre style='margin-top:8px; white-space:pre-wrap;'>{html.escape(event_lines(artifacts['events_json']))}</pre>"
                )
            )
        )


In [2]:
def on_render_clicked(_: widgets.Button) -> None:
    render_event_demo(Path(selector.value), force=force_rerun.value)


render_button.on_click(on_render_clicked)
controls = widgets.HBox([selector, force_rerun, render_button])
display(widgets.VBox([controls, ui_output]))
render_event_demo(Path(selector.value), force=False)
